# SmolVLA Server on Google Colab

FastAPI server hosting LeRobot's SmolVLA (450M param VLA) for remote 3-DOF arm control.

**Quick Start:**
1. Run all cells in order
2. Copy ngrok URL from final cell
3. Use in local client: `SMOLVLA_URL="https://...-ngrok.io"`
4. Keep notebook running during experiments

## 1. Install Dependencies

In [ ]:
%pip install -q "lerobot[smolvla]" torch torchvision uvicorn fastapi pydantic pyngrok aiohttp pillow requests 2>&1 | tail -3
print("✓ Dependencies installed")

## 2. Setup GPU

In [ ]:
import torch
import numpy as np

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("  ⚠ CPU mode (slow)")

## 3. Load SmolVLA Model

In [ ]:
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
from lerobot.policies.factory import make_pre_post_processors
import time

print("Loading SmolVLA...")
start = time.time()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_id = "lerobot/smolvla_base"

policy = SmolVLAPolicy.from_pretrained(model_id).to(device).eval()
preprocess, postprocess = make_pre_post_processors(
    policy.config,
    model_id,
    preprocessor_overrides={"device_processor": {"device": str(device)}},
)

elapsed = time.time() - start
print(f"✓ Model loaded in {elapsed:.1f}s")
print(f"  Params: {sum(p.numel() for p in policy.parameters()) / 1e6:.0f}M")
print(f"  Device: {device}")

## 4. Create FastAPI Endpoint

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List
from PIL import Image
import io
import base64

app = FastAPI(title="SmolVLA 3-DOF Arm Server")

class PredictRequest(BaseModel):
    rgb_image_b64: str

class PredictResponse(BaseModel):
    action: List[float]
    action_std: List[float]
    latency_ms: float

@app.get("/health")
async def health_check():
    return {"status": "ok", "model": "smolvla_base"}

@app.post("/predict", response_model=PredictResponse)
async def predict(request: PredictRequest):
    """Predict action from base64 RGB image."""
    start = time.time()
    
    try:
        # Decode image
        image_data = base64.b64decode(request.rgb_image_b64)
        pil_image = Image.open(io.BytesIO(image_data)).convert('RGB')
        image_array = np.array(pil_image, dtype=np.uint8)
        image_chw = np.transpose(image_array, (2, 0, 1))
        
        # Frame dict for LeRobot
        state_vector = np.zeros(6, dtype=np.float32)
        frame = {
            'observation.images.camera1': image_chw,
            'observation.images.camera2': image_chw,
            'observation.images.camera3': image_chw,
            'observation.state': state_vector,
            'task': 'reaching',
        }
        
        # Preprocess
        batch = preprocess(frame)
        
        # Fix dimensions (only 3D ndarrays need unsqueeze)
        for key, val in batch.items():
            if isinstance(val, np.ndarray) and val.ndim == 3:
                batch[key] = torch.from_numpy(val).unsqueeze(0).to(device)
            elif isinstance(val, torch.Tensor):
                batch[key] = val.to(device)
        
        # Inference
        with torch.inference_mode():
            action = policy.select_action(batch)
        
        # Postprocess and extract
        action_dict = postprocess({"action": action})
        action_denorm = action_dict["action"][0].cpu().numpy()
        action_4d = action_denorm[:4].tolist()
        
        elapsed = (time.time() - start) * 1000
        return PredictResponse(
            action=action_4d,
            action_std=[0.1, 0.1, 0.1, 0.2],
            latency_ms=elapsed
        )
    
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"{type(e).__name__}: {str(e)}")

print("✓ API endpoint configured")
print("  Routes: /health, /predict")

## 5. Setup ngrok Token

In [ ]:
from pyngrok import ngrok

# Set your ngrok auth token (get from https://dashboard.ngrok.com/get-started/your-authtoken)
ngrok_token = "3AtHvDrRiyj5PJCcRe03ogfhSzW_5VC6EQS8bcB2Rdz7oMbWv"

if ngrok_token != "<YOUR_TOKEN>":
    ngrok.set_auth_token(ngrok_token)
    print("✓ ngrok token configured")
else:
    print("⚠ ngrok token not set (local access only)")

## 6. Start Server & ngrok Tunnel

In [ ]:
import asyncio
import threading
from uvicorn import Config, Server

# Start server on port 8001
config = Config(app=app, host="0.0.0.0", port=8001, log_level="info")
server = Server(config)

def run_server():
    asyncio.run(server.serve())

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

import time
time.sleep(2)
print("✓ Server running on port 8001")

## 7. Expose via ngrok

In [ ]:
try:
    public_url = ngrok.connect(8001, "http")
    smolvla_url = str(public_url)
    
    print("\n" + "="*70)
    print("🌐 PUBLIC SmolVLA SERVER URL")
    print("="*70)
    print(f"\n{smolvla_url}")
    print("\n" + "="*70)
    print("\nUsage in local client:")
    print(f'  SMOLVLA_URL = "{smolvla_url}"')
    print("\nEndpoints:")
    print(f"  Health:  {smolvla_url}/health")
    print(f"  Predict: {smolvla_url}/predict")
    
except Exception as e:
    print(f"⚠ ngrok error: {e}")
    smolvla_url = "http://localhost:8001"
    print(f"Local-only: {smolvla_url}")

## 8. Test Endpoints

In [ ]:
import requests

print("\nTesting API...\n")

# Test health
try:
    resp = requests.get(f"http://localhost:8001/health", timeout=5)
    print(f"✓ /health: {resp.status_code}")
except Exception as e:
    print(f"✗ /health: {e}")

# Test predict
try:
    test_rgb = np.random.randint(0, 256, (224, 224, 3), dtype=np.uint8)
    test_img = Image.fromarray(test_rgb)
    buf = io.BytesIO()
    test_img.save(buf, format="PNG")
    test_b64 = base64.b64encode(buf.getvalue()).decode()
    
    resp = requests.post(
        "http://localhost:8001/predict",
        json={"rgb_image_b64": test_b64},
        timeout=20
    )
    
    if resp.status_code == 200:
        result = resp.json()
        print(f"✓ /predict: {resp.status_code}")
        print(f"  Action: {[round(a, 4) for a in result['action']]}")
        print(f"  Latency: {result['latency_ms']:.1f} ms")
    else:
        print(f"✗ /predict: {resp.status_code}")
        print(f"  Error: {resp.text[:200]}")
except Exception as e:
    print(f"✗ /predict: {e}")

print("\n✓ SmolVLA server is ready!")

## 9. Keep Server Running

The server will stay online while this cell runs. You can leave it running or use ngrok web dashboard to monitor.

In [ ]:
print(f"\nServer online at: {smolvla_url}")
print("Press Ctrl+C to stop (but keep notebook running in Colab)\n")

try:
    while True:
        time.sleep(30)
except KeyboardInterrupt:
    print("\nServer stopped by user")